# 002 — Análisis de Correlación entre Indicadores ODS de México

Este notebook analiza las correlaciones entre los indicadores consolidados de los ODS
para identificar **problemáticas interconectadas** y posibles soluciones.

**Fuente**: datos consolidados generados por `001_dataWrangling.ipynb`

## 1. Configuración y carga de datos

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

CONSOLIDATED_DIR = '../data/consolidated'


In [2]:
# Cargar tabla pivote (Periodo × Indicadores)
pivot_df = pd.read_csv(f'{CONSOLIDATED_DIR}/indicadores_nacionales.csv', index_col=0)
pivot_df.index = pivot_df.index.astype(int)

# Cargar metadatos
meta_df = pd.read_csv(f'{CONSOLIDATED_DIR}/metadatos_indicadores.csv')

print(f"📊 Datos cargados: {pivot_df.shape[0]} años × {pivot_df.shape[1]} indicadores")
print(f"   Rango temporal: {pivot_df.index.min()} - {pivot_df.index.max()}")

# Filtrar indicadores con al menos 5 observaciones para correlaciones válidas
MIN_OBS = 5
valid_indicators = pivot_df.columns[pivot_df.notna().sum() >= MIN_OBS].tolist()
pivot_valid = pivot_df[valid_indicators]
print(f"   Indicadores con ≥{MIN_OBS} observaciones: {len(valid_indicators)}")

📊 Datos cargados: 52 años × 292 indicadores
   Rango temporal: 1970 - 2026
   Indicadores con ≥5 observaciones: 199


## 2. Función de correlación con significancia estadística

In [3]:
def compute_pairwise_correlations(df, method='spearman', min_shared=5):
    """
    Calcula correlaciones entre todos los pares de columnas con validación estadística.
    
    Retorna un DataFrame con: ind_1, ind_2, correlation, p_value, n_shared
    """
    cols = df.columns.tolist()
    results = []
    
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            col_a, col_b = cols[i], cols[j]
            # Obtener datos compartidos (ambos no-NaN)
            mask = df[col_a].notna() & df[col_b].notna()
            n_shared = mask.sum()
            
            if n_shared < min_shared:
                continue
            
            x = df.loc[mask, col_a].values
            y = df.loc[mask, col_b].values
            
            if method == 'spearman':
                corr, pval = stats.spearmanr(x, y)
            else:
                corr, pval = stats.pearsonr(x, y)
            
            if np.isnan(corr):
                continue
            
            results.append({
                'ind_1': col_a,
                'ind_2': col_b,
                'correlation': corr,
                'p_value': pval,
                'n_shared': n_shared
            })
    
    return pd.DataFrame(results)

## 3. Cálculo de correlaciones (Spearman)

Usamos **Spearman** por ser más robusto ante relaciones no lineales y datos con
escalas muy diferentes entre indicadores.

In [4]:
print("⏳ Calculando correlaciones Spearman entre pares de indicadores...")
corr_results = compute_pairwise_correlations(pivot_valid, method='spearman', min_shared=MIN_OBS)
print(f"✅ {len(corr_results):,} pares calculados")

# Filtrar significativos
corr_significant = corr_results[corr_results['p_value'] < 0.05].copy()
print(f"   Pares significativos (p < 0.05): {len(corr_significant):,}")

# Alta correlación
corr_high = corr_significant[corr_significant['correlation'].abs() >= 0.7].copy()
print(f"   Pares con |r| ≥ 0.7: {len(corr_high):,}")

corr_very_high = corr_significant[corr_significant['correlation'].abs() >= 0.85].copy()
print(f"   Pares con |r| ≥ 0.85: {len(corr_very_high):,}")

⏳ Calculando correlaciones Spearman entre pares de indicadores...
✅ 14,984 pares calculados
   Pares significativos (p < 0.05): 4,799
   Pares con |r| ≥ 0.7: 3,758
   Pares con |r| ≥ 0.85: 2,500


## 4. Enriquecer resultados con metadatos

In [5]:
def enrich_correlations(corr_df, meta_df):
    """Agrega nombres de indicadores y ODS a los resultados de correlación."""
    meta_lookup = meta_df.set_index('indicator_id')[['title', 'ods_number']].to_dict('index')
    
    enriched = corr_df.copy()
    
    def get_meta(ind_id, field):
        if ind_id in meta_lookup:
            return meta_lookup[ind_id].get(field)
        return None
    
    enriched['nombre_1'] = enriched['ind_1'].apply(lambda x: get_meta(x, 'title'))
    enriched['nombre_2'] = enriched['ind_2'].apply(lambda x: get_meta(x, 'title'))
    enriched['ods_1'] = enriched['ind_1'].apply(lambda x: get_meta(x, 'ods_number'))
    enriched['ods_2'] = enriched['ind_2'].apply(lambda x: get_meta(x, 'ods_number'))
    
    # Truncar nombres largos para visualización
    enriched['nombre_corto_1'] = enriched['nombre_1'].apply(
        lambda x: (str(x)[:60] + '...') if x and len(str(x)) > 60 else x
    )
    enriched['nombre_corto_2'] = enriched['nombre_2'].apply(
        lambda x: (str(x)[:60] + '...') if x and len(str(x)) > 60 else x
    )
    
    return enriched

corr_enriched = enrich_correlations(corr_high, meta_df)

# Clasificar si la correlación es intra-ODS o inter-ODS
corr_enriched['cross_ods'] = corr_enriched['ods_1'] != corr_enriched['ods_2']
print(f"\nCorrelaciones altas intra-ODS: {(~corr_enriched['cross_ods']).sum()}")
print(f"Correlaciones altas inter-ODS (cruzadas): {corr_enriched['cross_ods'].sum()}")


Correlaciones altas intra-ODS: 434
Correlaciones altas inter-ODS (cruzadas): 3324


## 5. Top correlaciones positivas y negativas

In [6]:
# Top 20 correlaciones positivas
top_positive = corr_enriched.nlargest(20, 'correlation')
print("🔗 TOP 20 CORRELACIONES POSITIVAS (los indicadores que suben juntos):\n")
for _, row in top_positive.iterrows():
    print(f"  r={row['correlation']:.3f} | ODS {int(row['ods_1']) if pd.notna(row['ods_1']) else '?'} ↔ ODS {int(row['ods_2']) if pd.notna(row['ods_2']) else '?'}")
    print(f"    {row['ind_1']}: {row['nombre_corto_1']}")
    print(f"    {row['ind_2']}: {row['nombre_corto_2']}")
    print()

🔗 TOP 20 CORRELACIONES POSITIVAS (los indicadores que suben juntos):

  r=1.000 | ODS 1 ↔ ODS 11
    1.5.1: 1.5.1 Número de personas muertas, desaparecidas y afectadas ...
    11.5.1: 11.5.1 Número de personas muertas, desaparecidas y afectadas...

  r=1.000 | ODS 1 ↔ ODS 13
    1.5.1: 1.5.1 Número de personas muertas, desaparecidas y afectadas ...
    13.1.1: 13.1.1 Número de personas muertas, desaparecidas y afectadas...

  r=1.000 | ODS 10 ↔ ODS 16
    10.4.1: 10.4.1 Proporción laboral del PIB que comprende los salarios...
    16R.3A: 16r.3a Tasa de ocupación carcelaria federal

  r=1.000 | ODS 11 ↔ ODS 13
    11.5.1: 11.5.1 Número de personas muertas, desaparecidas y afectadas...
    13.1.1: 13.1.1 Número de personas muertas, desaparecidas y afectadas...

  r=1.000 | ODS 12 ↔ ODS 4
    12.2.2.A: 12.2.2.a Consumo material interno en términos absolutos
    4.A.1.G: 4.a.1.g Proporción de escuelas con lavabo de manos por entid...

  r=1.000 | ODS 12 ↔ ODS 5
    12.2.2.A: 12.2.2.a Consu

In [7]:
# Top 20 correlaciones negativas
top_negative = corr_enriched.nsmallest(20, 'correlation')
print("🔄 TOP 20 CORRELACIONES NEGATIVAS (cuando uno sube, el otro baja):\n")
for _, row in top_negative.iterrows():
    print(f"  r={row['correlation']:.3f} | ODS {int(row['ods_1']) if pd.notna(row['ods_1']) else '?'} ↔ ODS {int(row['ods_2']) if pd.notna(row['ods_2']) else '?'}")
    print(f"    {row['ind_1']}: {row['nombre_corto_1']}")
    print(f"    {row['ind_2']}: {row['nombre_corto_2']}")
    print()

🔄 TOP 20 CORRELACIONES NEGATIVAS (cuando uno sube, el otro baja):

  r=-1.000 | ODS 13 ↔ ODS 5
    13N.2.1: 13n.2.1 Intensidad de carbono por la quema de combustibles f...
    5N.2.1: 5n.2.1 Porcentaje de certificados y títulos expedidos a muje...

  r=-1.000 | ODS 13 ↔ ODS 8
    13N.2.1: 13n.2.1 Intensidad de carbono por la quema de combustibles f...
    8.5.1A: 8.5.1a Ingreso medio por hora de empleadas y empleados

  r=-1.000 | ODS 13 ↔ ODS 8
    13N.2.1: 13n.2.1 Intensidad de carbono por la quema de combustibles f...
    8.5.1B: 8.5.1b Ingreso medio por hora de empleadas y empleados, desg...

  r=-1.000 | ODS 13 ↔ ODS 8
    13N.2.1: 13n.2.1 Intensidad de carbono por la quema de combustibles f...
    8.5.1C: 8.5.1c Ingreso medio por hora de empleadas y empleados, desg...

  r=-1.000 | ODS 13 ↔ ODS 8
    13N.2.1: 13n.2.1 Intensidad de carbono por la quema de combustibles f...
    8.5.1D: 8.5.1d Ingreso medio por hora de empleadas y empleados, desg...

  r=-1.000 | ODS 16 ↔ ODS 9
    

## 6. Heatmap de correlación entre ODS

Promedio de las correlaciones absolutas entre indicadores cruzados por ODS.

In [8]:
# Crear matriz ODS × ODS con promedios de correlación
ods_names = {
    1: 'Pobreza', 2: 'Hambre', 3: 'Salud', 4: 'Educación',
    5: 'Género', 6: 'Agua', 7: 'Energía', 8: 'Trabajo',
    9: 'Industria', 10: 'Desigualdad', 11: 'Ciudades', 12: 'Producción',
    13: 'Clima', 14: 'Océanos', 15: 'Ecosistemas', 16: 'Paz', 17: 'Alianzas'
}

# Usar todas las correlaciones significativas (no solo las altas)
corr_all_enriched = enrich_correlations(corr_significant, meta_df)

# Filtrar indicadores con ODS asignado
corr_with_ods = corr_all_enriched.dropna(subset=['ods_1', 'ods_2'])

# Calcular promedio de |correlación| por par de ODS
ods_matrix_data = []
for _, row in corr_with_ods.iterrows():
    ods_a = int(row['ods_1'])
    ods_b = int(row['ods_2'])
    ods_matrix_data.append({'ods_a': ods_a, 'ods_b': ods_b, 'abs_corr': abs(row['correlation'])})
    ods_matrix_data.append({'ods_a': ods_b, 'ods_b': ods_a, 'abs_corr': abs(row['correlation'])})

ods_pairs_df = pd.DataFrame(ods_matrix_data)
ods_matrix = ods_pairs_df.pivot_table(index='ods_a', columns='ods_b', values='abs_corr', aggfunc='mean')

# Agregar la diagonal (auto-correlación = 1)
for ods in ods_matrix.index:
    if ods in ods_matrix.columns:
        ods_matrix.loc[ods, ods] = 1.0

# Renombrar
labels = [f"ODS {i}<br>{ods_names.get(i, '?')}" for i in ods_matrix.index]
ods_matrix.index = labels
ods_matrix.columns = labels

fig = px.imshow(
    ods_matrix,
    color_continuous_scale='YlOrRd',
    zmin=0, zmax=1,
    text_auto='.2f',
    aspect='auto',
    title='Correlación promedio absoluta entre ODS<br><sup>Basada en indicadores con correlación significativa (p < 0.05)</sup>',
    labels={'color': '|Correlación| promedio'}
)
fig.update_layout(height=650, margin=dict(l=10, r=10, t=60, b=10))
fig.update_xaxes(tickangle=-45)
fig.show()
print('✅ Heatmap interactivo generado')


✅ Heatmap interactivo generado


## 7. Conteo de correlaciones fuertes entre ODS

In [9]:
# Contar cuántas correlaciones fuertes hay entre cada par de ODS
corr_high_with_ods = corr_enriched.dropna(subset=['ods_1', 'ods_2']).copy()
corr_high_with_ods['ods_pair'] = corr_high_with_ods.apply(
    lambda r: tuple(sorted([int(r['ods_1']), int(r['ods_2'])])), axis=1
)

ods_pair_counts = corr_high_with_ods.groupby('ods_pair').agg(
    n_correlations=('correlation', 'count'),
    avg_corr=('correlation', 'mean'),
    max_corr=('correlation', lambda x: x.abs().max())
).reset_index()
ods_pair_counts.columns = ['ods_pair', 'n_correlaciones', 'corr_promedio', 'max_|corr|']
ods_pair_counts = ods_pair_counts.sort_values('n_correlaciones', ascending=False)
ods_pair_counts['ODS_A'] = ods_pair_counts['ods_pair'].apply(lambda x: f"ODS {x[0]} ({ods_names.get(x[0], '?')})")
ods_pair_counts['ODS_B'] = ods_pair_counts['ods_pair'].apply(lambda x: f"ODS {x[1]} ({ods_names.get(x[1], '?')})")

print("📊 PARES DE ODS CON MÁS CORRELACIONES FUERTES (|r|≥0.7, p<0.05):\n")
display_cols = ['ODS_A', 'ODS_B', 'n_correlaciones', 'corr_promedio', 'max_|corr|']
ods_pair_counts[display_cols].head(25)

📊 PARES DE ODS CON MÁS CORRELACIONES FUERTES (|r|≥0.7, p<0.05):



,ODS_A,ODS_B,n_correlaciones,corr_promedio,max_|corr|
37,ODS 3 (Salud),ODS 8 (Trabajo),177,-0.067895,1.000000
51,ODS 4 (Educación),ODS 8 (Trabajo),168,0.114340,1.000000
33,ODS 3 (Salud),ODS 4 (Educación),154,-0.051304,1.000000
32,ODS 3 (Salud),ODS 3 (Salud),139,0.008934,1.000000
59,ODS 4 (Educación),ODS 16 (Paz),133,0.295295,1.000000
45,ODS 3 (Salud),ODS 16 (Paz),110,-0.124672,1.000000
105,ODS 8 (Trabajo),ODS 16 (Paz),110,0.240915,1.000000
35,ODS 3 (Salud),ODS 6 (Agua),97,-0.005101,1.000000
76,ODS 6 (Agua),ODS 8 (Trabajo),88,-0.075222,1.000000
97,ODS 8 (Trabajo),ODS 8 (Trabajo),84,0.150979,1.000000


## 8. Scatter plots de los pares más relevantes

In [10]:
# Seleccionar los 9 pares más interesantes: alta correlación, diferentes ODS, diversidad
best_pairs = corr_enriched[corr_enriched['cross_ods']].copy()
best_pairs['abs_corr'] = best_pairs['correlation'].abs()

# Top 6 positivas + top 3 negativas cross-ODS
top_pos = best_pairs.nlargest(6, 'abs_corr')
top_neg = best_pairs[best_pairs['correlation'] < 0].nsmallest(3, 'correlation')
scatter_pairs = pd.concat([top_pos, top_neg]).drop_duplicates(subset=['ind_1', 'ind_2'])

n_pairs = min(len(scatter_pairs), 9)
if n_pairs > 0:
    ncols = 3
    nrows = (n_pairs + ncols - 1) // ncols

    # Crear títulos de subplots
    subplot_titles = []
    for _, row in scatter_pairs.head(9).iterrows():
        subplot_titles.append(f"r={row['correlation']:.3f} | ODS {int(row['ods_1'])} ↔ ODS {int(row['ods_2'])}")

    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=subplot_titles,
                        horizontal_spacing=0.12, vertical_spacing=0.15)

    for idx, (_, row) in enumerate(scatter_pairs.head(9).iterrows()):
        r, c = divmod(idx, ncols)
        mask = pivot_valid[row['ind_1']].notna() & pivot_valid[row['ind_2']].notna()
        x = pivot_valid.loc[mask, row['ind_1']]
        y = pivot_valid.loc[mask, row['ind_2']]
        years = pivot_valid.index[mask]

        color = 'green' if row['correlation'] > 0 else 'red'

        # Scatter con año en hover
        fig.add_trace(
            go.Scatter(
                x=x, y=y,
                mode='markers+text',
                text=[str(yr) if (yr % 5 == 0 or yr == years.max() or yr == years.min()) else '' for yr in years],
                textposition='top center',
                customdata=list(years),
                hovertemplate=f"%{{x:.3f}} / %{{y:.3f}}<br>Año: %{{customdata}}<extra>{row['ind_1']} vs {row['ind_2']}</extra>",
                marker=dict(size=8, color=color, opacity=0.7,
                            line=dict(color='black', width=0.5)),
                showlegend=False
            ),
            row=r+1, col=c+1
        )

        # Línea de tendencia
        z = np.polyfit(x, y, 1)
        p = np.poly1d(z)
        x_line = np.linspace(x.min(), x.max(), 100)
        fig.add_trace(
            go.Scatter(
                x=x_line, y=p(x_line),
                mode='lines',
                line=dict(dash='dash', color=color, width=1.5),
                opacity=0.5,
                showlegend=False,
                hoverinfo='skip'
            ),
            row=r+1, col=c+1
        )

        fig.update_xaxes(title_text=row['ind_1'], title_font=dict(size=9), row=r+1, col=c+1)
        fig.update_yaxes(title_text=row['ind_2'], title_font=dict(size=9), row=r+1, col=c+1)

    fig.update_layout(
        height=400 * nrows,
        title_text='Pares de indicadores con correlación cruzada más fuerte (ODS distintos)',
        title_font=dict(size=14),
        margin=dict(l=20, r=20, t=80, b=20)
    )
    fig.show()
    print('✅ Scatter plots interactivos generados')


✅ Scatter plots interactivos generados


## 9. Red de conexiones entre ODS

In [11]:
import math

# Colores oficiales por ODS
ods_colors = {
    1: '#E5243B', 2: '#DDA63A', 3: '#4C9F38', 4: '#C5192D',
    5: '#FF3A21', 6: '#26BDE2', 7: '#FCC30B', 8: '#A21942',
    9: '#FD6925', 10: '#DD1367', 11: '#FD9D24', 12: '#BF8B2E',
    13: '#3F7E44', 14: '#0A97D9', 15: '#56C02B', 16: '#00689D',
    17: '#19486A'
}

# Posiciones circulares para los 17 ODS
n_ods = 17
pos = {i: (math.cos(2 * math.pi * (i-1) / n_ods),
           math.sin(2 * math.pi * (i-1) / n_ods))
       for i in range(1, n_ods + 1)}

# Filtrar aristas con ≥2 correlaciones fuertes
edges_df = ods_pair_counts[ods_pair_counts['n_correlaciones'] >= 2].copy()
max_w = edges_df['n_correlaciones'].max() if not edges_df.empty else 1

# Grado ponderado por nodo
node_degree = {i: 0 for i in range(1, n_ods + 1)}
for _, row in edges_df.iterrows():
    a, b = row['ods_pair']
    node_degree[a] += row['n_correlaciones']
    node_degree[b] += row['n_correlaciones']

# Trazas de aristas
edge_traces = []
for _, row in edges_df.iterrows():
    a, b = row['ods_pair']
    x0, y0 = pos[a]
    x1, y1 = pos[b]
    width = 1 + (row['n_correlaciones'] / max_w) * 7
    mid_x = (x0 + x1) / 2
    mid_y = (y0 + y1) / 2
    edge_traces.append(go.Scatter(
        x=[x0, x1, None], y=[y0, y1, None],
        mode='lines',
        line=dict(width=width, color='rgba(100,100,100,0.4)'),
        hoverinfo='none',
        showlegend=False
    ))
    # Etiqueta en el centro de la arista
    edge_traces.append(go.Scatter(
        x=[mid_x], y=[mid_y],
        mode='text',
        text=[str(int(row['n_correlaciones']))],
        textfont=dict(size=8, color='gray'),
        hoverinfo='none',
        showlegend=False
    ))

# Traza de nodos
node_x = [pos[i][0] for i in range(1, n_ods + 1)]
node_y = [pos[i][1] for i in range(1, n_ods + 1)]
node_sizes = [2 + node_degree.get(i, 0) * 0.2 for i in range(1, n_ods + 1)]
node_colors_list = [ods_colors.get(i, '#CCCCCC') for i in range(1, n_ods + 1)]
node_hover = [f"<b>ODS {i}: {ods_names.get(i, '')}</b><br>Conexiones ponderadas: {node_degree.get(i, 0)}" for i in range(1, n_ods + 1)]
node_text = [f"ODS {i}" for i in range(1, n_ods + 1)]

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    text=node_text,
    textposition='middle center',
    textfont=dict(size=9, color='white', family='Arial Black'),
    hovertext=node_hover,
    hoverinfo='text',
    marker=dict(
        color=node_colors_list,
        size=node_sizes,
        line=dict(color='white', width=2)
    ),
    showlegend=False
)

fig = go.Figure(data=edge_traces + [node_trace])
fig.update_layout(
    title='Red de Conexiones entre ODS<br><sup>Aristas muestran cantidad de correlaciones fuertes (|r| ≥ 0.7)</sup>',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False,
               scaleanchor='x', scaleratio=1),
    height=700,
    margin=dict(l=20, r=20, t=60, b=20),
    plot_bgcolor='white',
    paper_bgcolor='#fafafa'
)
fig.show()
print('✅ Red interactiva generada')


✅ Red interactiva generada


## 10. Identificación de problemáticas interconectadas

Basándonos en las correlaciones, identificamos clusters de problemas que se refuerzan mutuamente.

In [12]:
# Análisis de clusters temáticos usando las correlaciones
print("=" * 80)
print("📋 PROBLEMÁTICAS INTERCONECTADAS IDENTIFICADAS")
print("=" * 80)

# Agrupar por par de ODS y mostrar los temas
cross_ods_top = corr_enriched[corr_enriched['cross_ods']].copy()
cross_ods_top['ods_pair_sorted'] = cross_ods_top.apply(
    lambda r: tuple(sorted([int(r['ods_1']), int(r['ods_2'])])), axis=1
)

# Para cada par de ODS con correlaciones fuertes, describir la problemática
for _, pair_row in ods_pair_counts.head(10).iterrows():
    ods_a, ods_b = pair_row['ods_pair']
    n = pair_row['n_correlaciones']
    
    print(f"\n{'─'*70}")
    print(f"🔗 ODS {ods_a} ({ods_names.get(ods_a, '?')}) ↔ ODS {ods_b} ({ods_names.get(ods_b, '?')})")
    print(f"   {n} correlaciones fuertes | Correlación promedio: {pair_row['corr_promedio']:.3f}")
    
    # Mostrar los 3 pares más fuertes de esta combinación
    pair_data = cross_ods_top[cross_ods_top['ods_pair_sorted'] == pair_row['ods_pair']]
    pair_data_sorted = pair_data.reindex(pair_data['correlation'].abs().sort_values(ascending=False).index)
    
    for _, detail in pair_data_sorted.head(3).iterrows():
        direction = "↑↑" if detail['correlation'] > 0 else "↑↓"
        print(f"   {direction} r={detail['correlation']:.3f}")
        print(f"      • {detail['ind_1']}: {detail['nombre_corto_1']}")
        print(f"      • {detail['ind_2']}: {detail['nombre_corto_2']}")

📋 PROBLEMÁTICAS INTERCONECTADAS IDENTIFICADAS

──────────────────────────────────────────────────────────────────────
🔗 ODS 3 (Salud) ↔ ODS 8 (Trabajo)
   177 correlaciones fuertes | Correlación promedio: -0.068
   ↑↓ r=-1.000
      • 3.7.2.B: 3.7.2.b Tasa de fecundidad en adolescentes (de 15 a 19 años)...
      • 8.5.1B: 8.5.1b Ingreso medio por hora de empleadas y empleados, desg...
   ↑↑ r=1.000
      • 3R.3A: 3r.3a Tasa de prevalencia del VIH/SIDA entre la población de...
      • 8.5.1A: 8.5.1a Ingreso medio por hora de empleadas y empleados
   ↑↑ r=1.000
      • 3R.3A: 3r.3a Tasa de prevalencia del VIH/SIDA entre la población de...
      • 8.5.1D: 8.5.1d Ingreso medio por hora de empleadas y empleados, desg...

──────────────────────────────────────────────────────────────────────
🔗 ODS 4 (Educación) ↔ ODS 8 (Trabajo)
   168 correlaciones fuertes | Correlación promedio: 0.114
   ↑↑ r=1.000
      • 4.5.1C: 4.5.1c Índice de paridad (entre mujeres y hombres) de la tas...
      • 8.5.

## 11. Exportar resultados

In [13]:
# Guardar correlaciones significativas
corr_export = corr_enriched[['ind_1', 'ind_2', 'correlation', 'p_value', 'n_shared',
                              'nombre_1', 'nombre_2', 'ods_1', 'ods_2', 'cross_ods']].copy()
corr_export.to_csv(f'{CONSOLIDATED_DIR}/correlaciones_altas.csv', index=False, encoding='utf-8')
print(f"✅ Correlaciones altas guardadas: {len(corr_export)} pares")

# Guardar la tabla de pares ODS
ods_pair_counts.to_csv(f'{CONSOLIDATED_DIR}/pares_ods_correlaciones.csv', index=False, encoding='utf-8')
print(f"✅ Resumen por pares de ODS guardado")

✅ Correlaciones altas guardadas: 3758 pares
✅ Resumen por pares de ODS guardado


In [15]:
print("\n" + "=" * 80)
print("📊 RESUMEN FINAL DEL ANÁLISIS DE CORRELACIÓN")
print("=" * 80)
print(f"\n   Indicadores analizados: {len(valid_indicators)}")
print(f"   Pares de correlaciones calculados: {len(corr_results):,}")
print(f"   Pares significativos (p < 0.05): {len(corr_significant):,}")
print(f"   Pares con |r| ≥ 0.7: {len(corr_high):,}")
print(f"   Pares inter-ODS con |r| ≥ 0.7: {corr_enriched['cross_ods'].sum()}")
print(f"\n   Visualizaciones generadas:")
print(f"     • Heatmap de correlación entre ODS")
print(f"     • Scatter plots de pares más fuertes")
print(f"     • Red de conexiones entre ODS")
print(f"\n   Archivos exportados:")
print(f"     • data/consolidated/correlaciones_altas.csv")
print(f"     • data/consolidated/pares_ods_correlaciones.csv")



📊 RESUMEN FINAL DEL ANÁLISIS DE CORRELACIÓN

   Indicadores analizados: 199
   Pares de correlaciones calculados: 14,984
   Pares significativos (p < 0.05): 4,799
   Pares con |r| ≥ 0.7: 3,758
   Pares inter-ODS con |r| ≥ 0.7: 3324

   Visualizaciones generadas:
     • Heatmap de correlación entre ODS
     • Scatter plots de pares más fuertes
     • Red de conexiones entre ODS

   Archivos exportados:
     • data/consolidated/correlaciones_altas.csv
     • data/consolidated/pares_ods_correlaciones.csv
